# Run the ML Pipeline

This notebook runs the full ML pipeline described in `PIPELINE.md`.

You can run the whole notebook from top to bottom, or run individual stage cells when debugging. Each stage uses the existing Python modules under `src/`, so the notebook is just an orchestration layer.

## 1. Setup

Run this cell first. It moves the notebook process to the `apps/ml-pipeline` directory and defines helpers for running each pipeline stage.

If your notebook kernel is already using the `.venv` for this folder, leave `INSTALL_REQUIREMENTS = False`. If not, set it to `True` once to install dependencies from `requirements.txt` into the active kernel environment.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_ml_pipeline_dir(start: Path | None = None) -> Path:
    """Find the apps/ml-pipeline directory from the current notebook location."""
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]

    for base in candidates:
        if (base / "PIPELINE.md").exists() and (base / "src").exists():
            return base
        nested = base / "apps" / "ml-pipeline"
        if (nested / "PIPELINE.md").exists() and (nested / "src").exists():
            return nested

    raise RuntimeError("Could not find apps/ml-pipeline. Open this notebook from the repo or ml-pipeline folder.")


ML_PIPELINE_DIR = find_ml_pipeline_dir()
os.chdir(ML_PIPELINE_DIR)

print(f"Using ML pipeline directory: {ML_PIPELINE_DIR}")
print(f"Python executable: {sys.executable}")

In [ ]:
# Keep this True for hackathon use: notebooks often run with a different Python than .venv.
INSTALL_REQUIREMENTS = True

REQUIRED_IMPORTS = [
    "pandas",
    "pyarrow",
    "openpyxl",
    "xlsxwriter",
    "matplotlib",
    "seaborn",
    "plotly",
    "sklearn",
    "lightgbm",
    "statsmodels",
    "mlflow",
]

missing = []
for package in REQUIRED_IMPORTS:
    try:
        __import__(package)
    except ImportError:
        missing.append(package)

if missing and INSTALL_REQUIREMENTS:
    print("Missing packages:", ", ".join(missing))
    print("Installing requirements.txt into this notebook kernel environment...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
elif missing:
    raise RuntimeError(
        "Missing packages: " + ", ".join(missing) +
        ". Set INSTALL_REQUIREMENTS = True or switch the notebook kernel to apps/ml-pipeline/.venv."
    )
else:
    print("All required pipeline packages are already available.")

## 2. Pipeline Runner Helpers

The helper below streams command output live inside the notebook and stops immediately if a stage fails.

In [ ]:
STAGES = [
    {
        "id": 1,
        "name": "Inspect raw tri-gen Excel files",
        "module": "src.data.inspect_tri_gen",
        "outputs": ["../../data/processed/tri-gen/_inspection_report.md", "../../data/processed/tri-gen/_inspection_report.json"],
    },
    {
        "id": 2,
        "name": "Clean tri-gen data to canonical long format",
        "module": "src.data.clean_tri_gen",
        "outputs": ["../../data/processed/tri-gen/long.parquet", "../../data/processed/tri-gen/_build_manifest.json", "../../data/processed/tri-gen/_cleaning_log.txt"],
    },
    {
        "id": 3,
        "name": "Build calendar, lag, and rolling features",
        "module": "src.features.build_features",
        "outputs": ["../../data/processed/tri-gen/features.parquet", "../../data/processed/tri-gen/_features_manifest.json"],
    },
    {
        "id": 4,
        "name": "Run EDA report",
        "module": "src.eda.run_eda",
        "outputs": ["reports/eda/index.html"],
    },
    {
        "id": 5,
        "name": "Train forecasters",
        "module": "src.training.train_forecaster",
        "outputs": ["../../checkpoints/forecaster/_summary.csv"],
    },
    {
        "id": 6,
        "name": "Train anomaly detectors",
        "module": "src.training.train_anomaly",
        "outputs": ["../../checkpoints/anomaly/_summary.csv"],
    },
    {
        "id": 7,
        "name": "Evaluate models and render report",
        "module": "src.evaluation.evaluate",
        "outputs": ["reports/eval/index.html", "reports/eval/forecaster_leaderboard.csv", "reports/eval/anomaly_leaderboard.csv"],
    },
]


def run_command(command: list[str], cwd: Path = ML_PIPELINE_DIR) -> None:
    """Run a command and stream stdout/stderr into the notebook."""
    print("$ " + " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}: {' '.join(command)}")


def run_stage(stage: dict) -> None:
    """Run one pipeline stage by module name."""
    print("\n" + "=" * 90)
    print(f"Stage {stage['id']}: {stage['name']}")
    print("=" * 90)
    start = time.time()

    run_command([sys.executable, "-m", stage["module"]])

    elapsed = time.time() - start
    print(f"\nDone in {elapsed:.1f}s")
    for rel in stage.get("outputs", []):
        path = (ML_PIPELINE_DIR / rel).resolve()
        status = "ok" if path.exists() else "missing"
        print(f"  [{status}] {path}")


def run_pipeline(start_stage: int = 1, end_stage: int = 7) -> None:
    """Run stages inclusively, stopping at the first error."""
    selected = [s for s in STAGES if start_stage <= s["id"] <= end_stage]
    if not selected:
        raise ValueError(f"No stages selected for range {start_stage}..{end_stage}")

    for stage in selected:
        run_stage(stage)

    print("\nFull selected pipeline completed.")


for stage in STAGES:
    print(f"{stage['id']}. {stage['name']} -> python -m {stage['module']}")

## 3. Run the Full Pipeline

Run this cell to execute all seven stages in order. If a stage fails, fix that stage and rerun from it using the individual stage cells below.

In [ ]:
run_pipeline(start_stage=1, end_stage=7)

## 4. Run Individual Stages

Use these cells when you want to inspect outputs between stages or resume after fixing an issue.

In [ ]:
# Stage 1: inspect raw Excel workbooks
run_stage(STAGES[0])

In [ ]:
# Stage 2: clean to canonical long.parquet
run_stage(STAGES[1])

In [ ]:
# Stage 3: build features.parquet
run_stage(STAGES[2])

In [ ]:
# Stage 4: run EDA report
run_stage(STAGES[3])

In [ ]:
# Stage 5: train LightGBM forecasters
run_stage(STAGES[4])

In [ ]:
# Stage 6: train anomaly detectors
run_stage(STAGES[5])

In [ ]:
# Stage 7: evaluate and render final report
run_stage(STAGES[6])

## 5. Inspect Outputs

After the pipeline runs, use these cells to open or inspect the most useful artifacts.

In [ ]:
from IPython.display import display

output_paths = [
    ML_PIPELINE_DIR / "../../data/processed/tri-gen/_inspection_report.md",
    ML_PIPELINE_DIR / "../../data/processed/tri-gen/_cleaning_log.txt",
    ML_PIPELINE_DIR / "../../data/processed/tri-gen/_features_manifest.json",
    ML_PIPELINE_DIR / "reports/eda/index.html",
    ML_PIPELINE_DIR / "reports/eval/index.html",
    ML_PIPELINE_DIR / "reports/eval/forecaster_leaderboard.csv",
    ML_PIPELINE_DIR / "reports/eval/anomaly_leaderboard.csv",
]

for path in output_paths:
    resolved = path.resolve()
    print(("ok      " if resolved.exists() else "missing ") + str(resolved))

In [ ]:
# Open the final evaluation report in your browser if it exists.
import webbrowser

eval_report = (ML_PIPELINE_DIR / "reports/eval/index.html").resolve()
if eval_report.exists():
    webbrowser.open(eval_report.as_uri())
    print(f"Opened {eval_report}")
else:
    print(f"Evaluation report not found yet: {eval_report}")

## 6. Optional: MLflow UI

Run this from a terminal if you want to browse model runs:

```powershell
cd D:\\Hackathons\\NRTF3\\repo\\nrtf\\apps\\ml-pipeline
mlflow ui --backend-store-uri file:./experiments
```

Then open `http://127.0.0.1:5000`.